# Задание 3 (основная часть). Полный пайплайн распознавания лиц

Собираем три стадии в один объект:

> **детекция** (готовый детектор) → **выравнивание** (наш Hourglass из Задания 1) → **распознавание** (наша сеть из Задания 2)

На вход — произвольная фотография (возможно, с несколькими лицами), на выходе — **эмбеддинг для
каждого найденного лица**. В конце демонстрируем работу: считаем косинусное сходство между
эмбеддингами лиц **одного** и **разных** людей.

**Правила проекта:** детектор можно брать готовый, предобученный на лицах (стадия 1). А вот
стадии 2 и 3 — это **наши** модели из Заданий 1 и 2.

## 0. Зависимости и веса моделей

Детектор — `MTCNN` из `facenet-pytorch` (быстрый и популярный). Мы используем его **только для
получения bounding box'ов** лиц; ключевые точки находит наша сеть из Задания 1, как требует задание.

In [ ]:
# !pip -q install facenet-pytorch

import os, math
import numpy as np, cv2
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from facenet_pytorch import MTCNN

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK_DIR = "/content/drive/MyDrive/face_project"
INPUT_SIZE, HEATMAP_SIZE, ALIGNED_SIZE, EMB_DIM = 256, 64, 112, 512
MEAN = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
STD  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
print("device:", DEVICE)

## 1. Повторяем определения моделей из Заданий 1–2

Ноутбук самодостаточен, поэтому повторяем компактные определения `StackedHourglass`,
`FaceEmbeddingNet`, функции декодирования точек и выравнивания, после чего загружаем
сохранённые веса.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.skip = nn.Identity() if ci==co else nn.Conv2d(ci, co, 1)
        self.conv1, self.bn1 = nn.Conv2d(ci, co//2, 1), nn.BatchNorm2d(co//2)
        self.conv2, self.bn2 = nn.Conv2d(co//2, co//2, 3, padding=1), nn.BatchNorm2d(co//2)
        self.conv3, self.bn3 = nn.Conv2d(co//2, co, 1), nn.BatchNorm2d(co)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        r = self.skip(x)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        return self.relu(self.bn3(self.conv3(x)) + r)

class Hourglass(nn.Module):
    def __init__(self, depth, ch):
        super().__init__()
        self.skip, self.pool = ResidualBlock(ch,ch), nn.MaxPool2d(2,2)
        self.before = ResidualBlock(ch,ch)
        self.inner  = Hourglass(depth-1, ch) if depth>1 else ResidualBlock(ch,ch)
        self.after  = ResidualBlock(ch,ch); self.up = nn.Upsample(scale_factor=2, mode="nearest")
    def forward(self, x):
        s = self.skip(x)
        return s + self.up(self.after(self.inner(self.before(self.pool(x)))))

class StackedHourglass(nn.Module):
    def __init__(self, num_stacks=2, num_landmarks=5, ch=256, depth=4):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3,64,7,2,3), nn.BatchNorm2d(64), nn.ReLU(True),
            ResidualBlock(64,128), nn.MaxPool2d(2,2), ResidualBlock(128,128), ResidualBlock(128,ch))
        self.hgs   = nn.ModuleList([Hourglass(depth,ch) for _ in range(num_stacks)])
        self.feats = nn.ModuleList([nn.Sequential(ResidualBlock(ch,ch), nn.Conv2d(ch,ch,1),
                        nn.BatchNorm2d(ch), nn.ReLU(True)) for _ in range(num_stacks)])
        self.heads = nn.ModuleList([nn.Conv2d(ch, num_landmarks, 1) for _ in range(num_stacks)])
        self.merge_feat = nn.ModuleList([nn.Conv2d(ch,ch,1) for _ in range(num_stacks-1)])
        self.merge_pred = nn.ModuleList([nn.Conv2d(num_landmarks,ch,1) for _ in range(num_stacks-1)])
    def forward(self, x):
        x = self.stem(x); outs = []
        for i in range(len(self.hgs)):
            feat = self.feats[i](self.hgs[i](x)); pred = self.heads[i](feat); outs.append(pred)
            if i < len(self.hgs)-1:
                x = x + self.merge_feat[i](feat) + self.merge_pred[i](pred)
        return outs

class FaceEmbeddingNet(nn.Module):
    def __init__(self, emb_dim=EMB_DIM, backbone="resnet50"):
        super().__init__()
        net = getattr(torchvision.models, backbone)(weights=None)
        in_feats = net.fc.in_features; net.fc = nn.Identity(); self.backbone = net
        self.embedding = nn.Sequential(nn.Linear(in_feats, emb_dim), nn.BatchNorm1d(emb_dim))
    def forward(self, x, normalize=False):
        emb = self.embedding(self.backbone(x))
        return F.normalize(emb) if normalize else emb

In [ ]:
REFERENCE_5PTS = np.array([[38.2946,51.6963],[73.5318,51.5014],[56.0252,71.7366],
                           [41.5493,92.3655],[70.7299,92.2041]], dtype=np.float32)

def decode_heatmaps(hm, input_size=INPUT_SIZE, heatmap_size=HEATMAP_SIZE):
    if torch.is_tensor(hm): hm = hm.detach().cpu().numpy()
    K, H, W = hm.shape; pts = np.zeros((K,2), np.float32)
    for k in range(K):
        y, x = np.unravel_index(hm[k].argmax(), (H, W)); pts[k] = [x, y]
    return pts * (input_size / heatmap_size)

def align_face(img, landmarks, out_size=ALIGNED_SIZE, ref=REFERENCE_5PTS):
    M, _ = cv2.estimateAffinePartial2D(np.asarray(landmarks, np.float32),
                                       ref * (out_size/112.0), method=cv2.LMEDS)
    return cv2.warpAffine(img, M, (out_size, out_size), borderValue=0)

# --- загрузка весов (из Заданий 1 и 2) ---
hourglass = StackedHourglass().to(DEVICE).eval()
hourglass.load_state_dict(torch.load(os.path.join(WORK_DIR, "hourglass_best.pt"), map_location=DEVICE))

embnet = FaceEmbeddingNet().to(DEVICE).eval()
embnet.load_state_dict(torch.load(os.path.join(WORK_DIR, "embnet_arcface.pt"), map_location=DEVICE))

detector = MTCNN(keep_all=True, device=DEVICE)   # стадия 1: только bbox'ы
print("Модели загружены.")

## 2. Класс `FacePipeline`

Логика для одного лица:

1. **Детекция** — MTCNN отдаёт bbox'ы всех лиц на фото.
2. **Кроп + точки** — расширяем bbox, кропаем, ресайзим до 256, прогоняем через `Hourglass`,
   декодируем 5 точек.
3. **Выравнивание** — similarity-transform к эталону → лицо `112×112`.
4. **Эмбеддинг** — `embnet` выдаёт нормализованный вектор.

In [ ]:
class FacePipeline:
    def __init__(self, detector, hourglass, embnet, margin=0.35):
        self.det, self.hg, self.emb, self.margin = detector, hourglass, embnet, margin

    def _landmarks(self, face_bgr):
        face = cv2.cvtColor(cv2.resize(face_bgr, (INPUT_SIZE, INPUT_SIZE)), cv2.COLOR_BGR2RGB)
        x = (torch.from_numpy(face).permute(2,0,1).float()/255.0 - MEAN)/STD
        with torch.no_grad():
            hm = self.hg(x.unsqueeze(0).to(DEVICE))[-1][0]
        return decode_heatmaps(hm), face

    def _embed(self, aligned_rgb):
        x = (torch.from_numpy(aligned_rgb).permute(2,0,1).float()/255.0 - MEAN)/STD
        with torch.no_grad():
            return self.emb(x.unsqueeze(0).to(DEVICE), normalize=True)[0].cpu().numpy()

    def __call__(self, image_bgr):
        '''-> список dict(bbox, aligned, embedding) для каждого найденного лица.'''
        rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        boxes, _ = self.det.detect(rgb)
        results = []
        if boxes is None:
            return results
        H, W = image_bgr.shape[:2]
        for (x1, y1, x2, y2) in boxes.astype(int):
            bw, bh = x2-x1, y2-y1
            dx, dy = int(bw*self.margin), int(bh*self.margin)
            xa, ya, xb, yb = max(0,x1-dx), max(0,y1-dy), min(W,x2+dx), min(H,y2+dy)
            crop = image_bgr[ya:yb, xa:xb]
            if crop.size == 0: continue
            pts, face_rgb = self._landmarks(crop)
            aligned = align_face(face_rgb, pts)
            results.append({"bbox": (x1,y1,x2,y2), "aligned": aligned,
                            "embedding": self._embed(aligned)})
        return results

pipe = FacePipeline(detector, hourglass, embnet)
print("Пайплайн готов: pipe(image_bgr) -> [{bbox, aligned, embedding}, ...]")

## 3. Демонстрация: один человек vs разные люди

Подготовьте несколько фотографий (можно свои). Идея проверки:

* два разных фото **одного** человека → косинусное сходство эмбеддингов должно быть **высоким**;
* фото **разных** людей → сходство **низкое**.

Так мы видим, что пространство эмбеддингов осмысленное и пайплайн работает целиком.

In [ ]:
def first_embedding(path):
    '''Берём первое (самое уверенное) лицо с фото и возвращаем его эмбеддинг и выровненное лицо.'''
    img = cv2.imread(path); res = pipe(img)
    assert res, f"Лицо не найдено: {path}"
    return res[0]["embedding"], res[0]["aligned"]

def cosine(a, b):
    return float(np.dot(a, b))    # эмбеддинги уже L2-нормализованы

# ⬇️ замените на свои файлы: два фото person A и одно фото person B
paths = {
    "A_photo1": os.path.join(WORK_DIR, "demo/personA_1.jpg"),
    "A_photo2": os.path.join(WORK_DIR, "demo/personA_2.jpg"),
    "B_photo1": os.path.join(WORK_DIR, "demo/personB_1.jpg"),
}
emb, faces = {}, {}
for k, p in paths.items():
    emb[k], faces[k] = first_embedding(p)

pairs = [("A_photo1","A_photo2","один человек"), ("A_photo1","B_photo1","разные люди")]
plt.figure(figsize=(10, 5))
for i,(k1,k2,label) in enumerate(pairs):
    s = cosine(emb[k1], emb[k2])
    plt.subplot(2,2,2*i+1); plt.imshow(faces[k1]); plt.axis("off"); plt.title(k1)
    plt.subplot(2,2,2*i+2); plt.imshow(faces[k2]); plt.axis("off")
    plt.title(f"{k2}\n{label}: cos = {s:.3f}")
plt.tight_layout(); plt.show()

In [ ]:
# Порог принятия решения «тот же человек?» можно подобрать (см. доп. задание ID-Rate).
THRESHOLD = 0.4
for k1, k2, label in pairs:
    s = cosine(emb[k1], emb[k2])
    verdict = "ОДИН человек" if s > THRESHOLD else "РАЗНЫЕ люди"
    print(f"{k1} ↔ {k2}: cos={s:.3f} -> {verdict}  (истина: {label})")

## Итоги

* Собрали сквозной пайплайн `FacePipeline`: детектор (MTCNN) → наш `Hourglass` (точки + выравнивание) → наш `FaceEmbeddingNet` (эмбеддинг).
* На вход — любое фото; на выходе — bbox, выровненное лицо и эмбеддинг для каждого лица.
* Показали, что сходство эмбеддингов разделяет «один человек / разные люди».

> Порог `THRESHOLD` лучше выбирать не на глаз, а по метрике из **доп. задания 1 (ID‑Rate)**:
> там мы фиксируем долю ложных срабатываний (FPR) и получаем соответствующий порог.